In [1]:
import gdown

In [8]:
gdown.download("https://drive.google.com/file/d/1fy4kC9GaM-vQudd3AQC_-gkcSag_0TJO/view?usp=drive_link", "premodels.rar", quiet=False, fuzzy=True)

Downloading...
From (original): https://drive.google.com/uc?id=1fy4kC9GaM-vQudd3AQC_-gkcSag_0TJO
From (redirected): https://drive.google.com/uc?id=1fy4kC9GaM-vQudd3AQC_-gkcSag_0TJO&confirm=t&uuid=7281942f-8b3e-4797-a3d9-f53157ef7f71
To: /path/to/FaceLinkGen/canfg/premodels.rar
100%|██████████| 2.66G/2.66G [00:41<00:00, 64.6MB/s]


'premodels.rar'

In [9]:
!unrar x premodels.rar


UNRAR 6.11 beta 1 freeware      Copyright (c) 1993-2022 Alexander Roshal


Extracting from premodels.rar

Creating    premodels                                                 OK
Extracting  premodels/anonymized_rec100_id_10_em_0_lp_0.pt                                  1 1 1 1 1 1 1 1 1 1 2 2 2 2 2 2 2 2 2 2 3 3 3 3 3 35  OK 
Extracting  premodels/anonymized_rec100_id_10_em_0_lp_0_G.pt            3 3 3 3 3 4 41  OK 
Extracting  premodels/irse.py                                           41  OK 
Extracting  premodels/irse50_seed85_anonymized_100_id_0_em_500_lp_10_EM.pt    4 4 4 4 4 4 47  OK 
Extracting  premodels/model_ir_se50.pth                                 4 4 4 5 5 5 53  OK 
Creating    premodels/premodels                                       OK
Extracting  premodels/premodels/irse.py                                 53  OK 
Extracting  premodels/seed85_anonymized_100_id_0_em_500_lp_10.pt        5 5 5 5 5 5 5 6 6 6 6 6 6 6 6 6 6 7 7 7 7 7 7 7 7 7 7 8 8 8 8 8 8 8 8 8 8 9 9 9 9 9

In [12]:
from deepface import DeepFace
import os
image = os.listdir("/path/to/casia-webface/008231/")[0]
img_path = os.path.join("/path/to/casia-webface/008231/", image)
face = DeepFace.extract_faces(img_path = img_path, detector_backend="mtcnn")

2026-02-02 21:48:05.406185: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1770097685.406350  577384 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 45847 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:3b:00.0, compute capability: 8.6
2026-02-02 21:48:05.406887: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1770097685.407102  577384 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 11347 MB memory:  -> device: 1, name: NVIDIA RTX A6000, pci bus id: 0000:af:00.0, compute capability: 8.6
2026-02-02 21:48:05.407593: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_valu

In [ ]:
import cv2
face = cv2.resize(face[0]['face'], (128, 128))

In [18]:
os.makedirs("./dataset/", exist_ok=True)
cv2.imwrite("./dataset/face.png", (face * 255).astype("uint8")[..., ::-1])

True

In [1]:
from deepface import DeepFace

2026-02-02 22:46:01.177407: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-02 22:46:01.245394: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-02 22:46:03.115741: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [23]:
import os
files = os.listdir("./CanFG/data/CelebA/protected_A/")

In [20]:
from io import BytesIO
import requests

def compare_faces(img1, img2, api_key, api_secret):
    buf1 = BytesIO()
    img1.save(buf1, format="JPEG")
    buf1.seek(0)

    buf2 = BytesIO()
    img2.convert("RGB").save(buf2, format="JPEG")
    buf2.seek(0)

    files = {
        "image_file1": ("img1.jpg", buf1, "image/jpeg"),
        "image_file2": ("img2.jpg", buf2, "image/jpeg")
    }

    data = {
        "api_key": api_key,
        "api_secret": api_secret
    }

    r = requests.post(
        "https://api-us.faceplusplus.com/facepp/v3/compare",
        data=data,
        files=files
    )
    return r.json()


In [24]:
import numpy as np
from PIL import Image
rate = []
for i in files:
    verified = compare_faces(Image.open("./CanFG/data/CelebA/protected_AA/"+i), Image.open("./dataset_processed/"+i.replace("png", "jpg")), "YOUR_FACEPP_API_KEY", "YOUR_FACEPP_API_SECRET")
    # verified = DeepFace.verify(im g1_path="./CanFG/data/CelebA/protected_A/"+i, img2_path = "./dataset_processed/"+i.replace("png", "jpg"), detector_backend="mtcnn", enforce_detection=False)["verified"]
    # print(verified)
    rate.append(verified["confidence"] > 74.0)
    print(np.mean(rate))

0.0
0.5
0.3333333333333333
0.25
0.2
0.16666666666666666
0.14285714285714285
0.125
0.1111111111111111


KeyError: 'confidence'

In [62]:
import pickle

with open(f"log/insight_student_embeddings_val_epoch{4}.pkl", "rb") as f:
    data = pickle.load(f)

In [63]:
data.keys()

dict_keys(['filenames', 'student_embeddings', 'teacher_embeddings', 'templates'])

In [64]:
import torch
filenames = sum(data["filenames"], [])
student_embeddings = torch.cat(data["student_embeddings"], axis=0)
teacher_embeddings = torch.cat(data["teacher_embeddings"], axis=0)

In [65]:
import random
random.seed(42)
import pandas as pd
map_ids = pd.read_csv('/path/to/Identity_CelebA.txt', sep=' ')

map_ids_dict = map_ids.to_dict(orient="list")
map_ids_dict = dict(zip(map_ids_dict["Image_name"], map_ids_dict["Label"]))

In [ ]:
with open("log/teacher_embeddings_insight.pkl", 'rb') as f:
    data2 = pickle.load(f)

In [75]:
id_embedding = {}
for i in range(len(filenames)):
    id = map_ids_dict[filenames[i].replace("CanFG/data/CelebA/protected_A/img_align_celeba_", "").replace(".png", ".jpg")]
    protected_embedding = data2[filenames[i]]
    id_embedding[id] = id_embedding.get(id, []) + [(student_embeddings[i], teacher_embeddings[i], protected_embedding)]

In [91]:
query = []
database = []
protected_embedding = []

for id, embeddings in id_embedding.items():
    student_embs = torch.stack([e[0] for e in embeddings], axis=0)
    teacher_embs = torch.stack([e[1] for e in embeddings], axis=0)
    teacher_embs = teacher_embs / teacher_embs.norm(dim=1, keepdim=True)
    student_embs = student_embs / student_embs.norm(dim=1, keepdim=True)
    if len(teacher_embs) < 2:
        continue
    sim = teacher_embs @ teacher_embs.T
    sim[torch.eye(len(sim), dtype=bool)] = 0
    valid_ids = torch.where(sim.max(dim=1).values > 0.25)[0]
    if valid_ids.sum() < 2:
        continue
    
    query.append(student_embs[valid_ids[1]])
    database.append(teacher_embs[valid_ids[1]])
    protected_embedding.append(torch.tensor(embeddings[valid_ids[1]][2]))

In [92]:
query = torch.stack(query, axis=0)
database = torch.stack(database, axis=0)
protected_embedding = torch.stack(protected_embedding, axis=0)

In [95]:
torch.nn.functional.cosine_similarity(query, database).mean()

tensor(0.5199)

In [88]:
query.shape, database.shape, protected_embedding.shape

(torch.Size([2744, 512]), torch.Size([2744, 512]), torch.Size([2744, 512]))

In [89]:
torch.mean(((protected_embedding @ database.T).argmax(dim=1) == torch.arange(len(query))).float())

tensor(0.0434)

In [90]:
for 

SyntaxError: invalid syntax (1235331270.py, line 1)